# M5A5 - Segmentação de Falhas em Tecidos

Na prática de hoje vamos refinar um modelo para a tarefa de segmentação de falhas em tecidos.

Esse notebook está estruturado da seguinte forma.

- Introdução
- Carregar Base de Dados
- Refinar Modelo
- Validar Modelo
- Próximos passos
- Atividade Complementares

## Introdução

Instalação para os que ainda não possuem a biblioteca instalada.

In [1]:
%pip install torch torchvision tqdm ipywidgets

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


Importar as bibliotecas

In [2]:
import torch
import torchvision
from tqdm.notebook import tqdm


## Carregar Base de Dados

A primeira tarefa para refinar um modelo é criar a base de dados.

Referência: https://docs.pytorch.org/vision/main/generated/torchvision.datasets.DTD.html

In [3]:
# Define transforms: Resize and normalize images as expected by most pre-trained models (e.g., ImageNet models use 224x224 input).
# Data augmentation is also crucial for better performance.
image_transforms = torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),
    torchvision.transforms.CenterCrop(224), # Common input size for ImageNet models
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load the dataset
# Set download=True to automatically fetch the dataset if not already present
train_dataset = torchvision.datasets.DTD(root="./data", split="train", download=True, transform=image_transforms)
val_dataset = torchvision.datasets.DTD(root="./data", split="val", download=True, transform=image_transforms)
test_dataset = torchvision.datasets.DTD(root="./data", split="test", download=True, transform=image_transforms)

# Define DataLoaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
_ = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)

 77%|███████▋  | 480M/625M [12:31<03:48, 638kB/s]   


KeyboardInterrupt: 

## Refinar Modelo

Na prática de hoje iremos refinar o modelo **ResNets** disponível no torchvision.

In [6]:
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)

num_classes = 47

# Get the number of input features for the final layer
in_features = model.fc.in_features

# Replace the existing fully connected layer with a new one
model.fc = torch.nn.Linear(in_features, num_classes)

# Move the model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = torch.nn.CrossEntropyLoss()

# Observe that all parameters are being optimized
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# Treinamento do modelo
model.train()
epochs = 100 # Alterar para treinar mais epocas.
for epoch in tqdm(range(epochs)):
    iteration = 0
    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if iteration % 100 == 0:
            print(f"Total loss: {loss.item()}")
        iteration += 1



  0%|          | 0/100 [00:00<?, ?it/s]

Total loss: 4.129215240478516
Total loss: 2.498401641845703
Total loss: 1.692678451538086
Total loss: 1.2997723817825317
Total loss: 1.028595209121704
Total loss: 1.004145860671997
Total loss: 0.5832077264785767
Total loss: 0.5114887356758118
Total loss: 0.4395548105239868
Total loss: 0.33516615629196167
Total loss: 0.2254084050655365
Total loss: 0.26441746950149536
Total loss: 0.20267868041992188
Total loss: 0.09835848212242126
Total loss: 0.11594346165657043
Total loss: 0.0841384008526802
Total loss: 0.08286389708518982
Total loss: 0.11256883293390274
Total loss: 0.08622539043426514
Total loss: 0.08969129621982574
Total loss: 0.05857635661959648
Total loss: 0.09913458675146103
Total loss: 0.0376826673746109
Total loss: 0.08283917605876923
Total loss: 0.07110682874917984
Total loss: 0.01985970139503479
Total loss: 0.03303223103284836
Total loss: 0.03441348671913147
Total loss: 0.02725941501557827
Total loss: 0.022226814180612564
Total loss: 0.03698214143514633
Total loss: 0.0331931784

## Validar Modelo

Agora vamos testar o nosso modelo.

In [7]:
num_correct = 0
num_samples = 0
    
# Set model to evaluation mode
model.eval()

# Disable gradient calculation for inference
with torch.no_grad():
    for images, labels in test_loader:
            x = images.to(device)
            y = labels.to(device)

            scores = model(x)

            # Get the index (class) with the highest score
            _, predictions = scores.max(1)
            
            # Count correct predictions
            num_correct += (predictions == y).sum().item()
            # Count total samples
            num_samples += predictions.size(0)

    # Calculate and print accuracy
    accuracy = float(num_correct) / float(num_samples) * 100
    print(f'Got {num_correct} / {num_samples} with accuracy {accuracy:.2f}%')


Got 1240 / 1880 with accuracy 65.96%


## Próximos Passos e Referências

Nas próximas práticas vamos continuar trabalhando com problemas reais que envolvem Visão Computacional.

Uma lista não exaustiva de referências segue:

- https://docs.pytorch.org/vision/main/generated/torchvision.datasets.DTD.html
- https://docs.pytorch.org/vision/main/models/generated/torchvision.models.resnet18.html
- https://huggingface.co/datasets
- https://pytorch.org/
- https://docs.pytorch.org/vision/main/models.html
- https://opencv.org/
- https://learnopencv.com/blogs/
- https://pyimagesearch.com/

## Atividades Complementares (Opcional)

- [ ] Tente por menos tempo faz com que a perfomance do modelo matenha-se a mesma?
- [ ] Tente alterar alguns hiperparâmetros de treinamento, batch e learning rate e veja como isso altera os resultados.